# Base model — greedy OlympiadBench evaluation

Evaluation only: no training. This notebook evaluates the base EXAONE model on `ChuGyouk/OlympiadBench-Math-Ko` with sampling explicitly disabled.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Korean-TDCS"
%cd /content
!if [ -d korean-math-tdcs/.git ]; then git -C korean-math-tdcs pull --ff-only; else git clone {REPO_URL} korean-math-tdcs; fi
%cd /content/korean-math-tdcs
%pip uninstall -y torchao
%pip install -e .

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_results = Path("/content/drive/MyDrive/Korean-TDCS/results")
local_results = Path.cwd() / "results"
drive_results.mkdir(parents=True, exist_ok=True)

if local_results.is_symlink():
    if local_results.resolve() != drive_results.resolve():
        raise RuntimeError(f"{local_results} points to the wrong Drive directory")
elif local_results.exists():
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    shutil.rmtree(local_results)

if not local_results.exists():
    local_results.symlink_to(drive_results, target_is_directory=True)

print(f"Saving all outputs to {drive_results}")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# ---- User controls ----
EVAL_BATCH_SIZE = 16
EVAL_MAX_TOKENS = 4096
RESULTS_PATH = Path(
    "results/greedy/base/olympiad_bench_math_ko/"
    f"max_tokens_{EVAL_MAX_TOKENS}/metrics.json"
)

print("Model: base EXAONE-4.0-1.2B")
print("Benchmark: ChuGyouk/OlympiadBench-Math-Ko (test)")
print("Decoding: greedy (do_sample=False)")
print(f"Results: {RESULTS_PATH}")

In [ ]:
cmd = ("python scripts/evaluate.py --config configs/baseline.yaml "
       f"--set evaluation.batch_size={EVAL_BATCH_SIZE} "
       f"--set evaluation.generation.max_new_tokens={EVAL_MAX_TOKENS} "
       "--set evaluation.generation.do_sample=false "
       f"--set output.results_path={RESULTS_PATH}")
print(cmd)
!{cmd}

In [ ]:
import json

import pandas as pd

result = json.load(RESULTS_PATH.open())
if result["generation"].get("do_sample") is not False:
    raise RuntimeError("Evaluation was not greedy")
pd.DataFrame({k: v for k, v in result["benchmarks"].items()}).T